In [15]:
from dotenv import load_dotenv

load_dotenv('../.env')

import anthropic

client = anthropic.Anthropic()

In [2]:
import json

system_prompt = "Suas respostas devem ser curtas, concisas e diretas ao ponto. Evite explicações longas e detalhadas."
historico = []
iteracao = 0

print("Chat iniciado. Digite 'sair' para encerrar.\n")

while True:
    entrada_usuario = input("Você: ")

    if entrada_usuario.lower() == "sair":
        break

    iteracao += 1

    historico.append({"role": "user", "content": entrada_usuario})

    response = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=1024,
        system=system_prompt,
        messages=historico,
    )

    texto_resposta = response.content[0].text

    historico.append({"role": "assistant", "content": texto_resposta})

    # Mostrar histórico com system prompt antes de cada user prompt (reenviado a cada iteração)
    historico_com_system = []
    for msg in historico:
        if msg["role"] == "user":
            historico_com_system.append({"role": "system", "content": system_prompt})
        historico_com_system.append(msg)

    print("\n" + f"------ Iteração {iteracao} ------".ljust(40, '-'))
    print(json.dumps(historico_com_system, indent=4))
    print(''.ljust(40, '-'))

Chat iniciado. Digite 'sair' para encerrar.


------ Iteração 1 ----------------------
[
    {
        "role": "system",
        "content": "Suas respostas devem ser curtas, concisas e diretas ao ponto. Evite explica\u00e7\u00f5es longas e detalhadas."
    },
    {
        "role": "user",
        "content": "Crie um arquivo .txt na raiz desse projeto com o conte\u00fado \"hello world\""
    },
    {
        "role": "assistant",
        "content": "N\u00e3o tenho acesso ao sistema de arquivos do seu computador. N\u00e3o posso criar, modificar ou deletar arquivos diretamente.\n\nPara criar o arquivo voc\u00ea mesmo, use:\n\n**No terminal (Linux/Mac):**\n```bash\necho \"hello world\" > hello.txt\n```\n\n**No terminal (Windows PowerShell):**\n```powershell\n\"hello world\" | Out-File -FilePath hello.txt -Encoding UTF8\n```\n\nOu simplesmente crie um arquivo `.txt` manualmente no editor de texto de sua prefer\u00eancia."
    }
]
----------------------------------------


In [16]:
def criar_arquivo_txt(nome_arquivo: str, conteudo: str) -> str:
    """Cria um arquivo .txt com o conteúdo informado"""
    try:
        with open(f"{nome_arquivo}.txt", "w", encoding="utf-8") as f:
            f.write(conteudo)
        return f"Arquivo '{nome_arquivo}.txt' criado com sucesso!"
    except Exception as e:
        return f"Erro ao criar arquivo: {str(e)}"


tool_criar_arquivo = {
    "name": "criar_arquivo_txt",
    "description": "Cria um arquivo de texto (.txt) com o conteúdo informado",
    "input_schema": {
        "type": "object",
        "properties": {
            "nome_arquivo": {
                "type": "string",
                "description": "Nome do arquivo a ser criado (sem a extensão .txt)"
            },
            "conteudo": {
                "type": "string",
                "description": "Conteúdo a ser escrito no arquivo"
            }
        },
        "required": ["nome_arquivo", "conteudo"]
    }
}

In [17]:
# Função para processar tool calls
def processar_tool_call(tool_name: str, tool_input: dict) -> str:
    """Processa a chamada de uma tool e retorna o resultado"""
    if tool_name == "criar_arquivo_txt":
        return criar_arquivo_txt(
            nome_arquivo=tool_input.get("nome_arquivo"),
            conteudo=tool_input.get("conteudo")
        )
    else:
        return f"Tool '{tool_name}' não reconhecida"

# Função para executar tool calls da resposta do Claude
def executar_tool_calls(response) -> list:
    """Extrai e executa todas as tool calls da resposta"""
    resultados = []
    
    for bloco in response.content:
        if bloco.type == "tool_use":          
            resultado = processar_tool_call(bloco.name, bloco.input)
            
            resultados.append({
                "tool_use_id": bloco.id,
                "nome_tool": bloco.name,
                "resultado": resultado
            })
    
    return resultados


In [18]:
import json

system_prompt = "Responda da forma mais breve possivel, sem rodeios, e de forma objetiva. Evite respostas longas e detalhadas."
tools = [tool_criar_arquivo]
historico = []
rodada = 0

print("Chat iniciado. Digite 'sair' para encerrar.\n")

while True:
    entrada_usuario = input("Você: ")

    if entrada_usuario.lower() == "sair":
        break

    historico.append({"role": "user", "content": entrada_usuario})

    # Loop interno: repete enquanto o Claude solicitar tool calls.
    # Cada volta desse loop é uma chamada (rodada) separada à API.
    while True:
        rodada += 1

        # Snapshot do que está sendo enviado NESTA rodada (histórico até aqui)
        enviado = {
            "system": system_prompt,
            "tools": tools,
            "messages": [dict(msg) for msg in historico],
        }

        print(f"\n{f'====== Rodada {rodada} — ENVIADO ======':=<50}")
        print(json.dumps(enviado, indent=4, ensure_ascii=False))

        response = client.messages.create(
            model="claude-haiku-4-5",
            max_tokens=1024,
            system=system_prompt,
            messages=historico,
            tools=tools,
        )

        recebido = {
            "stop_reason": response.stop_reason,
            "content": [bloco.model_dump() for bloco in response.content],
        }

        print(f"\n{f'------ Rodada {rodada} — RECEBIDO ------':-<50}")
        print(json.dumps(recebido, indent=4, ensure_ascii=False))
        print(''.ljust(50, '='))

        # Guarda a resposta completa do assistant (texto e/ou tool_use)
        historico.append({"role": "assistant", "content": recebido["content"]})

        if response.stop_reason != "tool_use":
            break

        # Chamada -> execução -> conclusão das tool calls
        resultados = executar_tool_calls(response)

        historico.append({
            "role": "user",
            "content": [
                {
                    "type": "tool_result",
                    "tool_use_id": r["tool_use_id"],
                    "content": r["resultado"],
                }
                for r in resultados
            ]
        })

    texto_resposta = next(
        (bloco.text for bloco in response.content if bloco.type == "text"),
        ""
    )
    print(f"\nClaude: {texto_resposta}")

Chat iniciado. Digite 'sair' para encerrar.


====== Rodada 1 — ENVIADO ========================
{
    "system": "Responda da forma mais breve possivel, sem rodeios, e de forma objetiva. Evite respostas longas e detalhadas.",
    "tools": [
        {
            "name": "criar_arquivo_txt",
            "description": "Cria um arquivo de texto (.txt) com o conteúdo informado",
            "input_schema": {
                "type": "object",
                "properties": {
                    "nome_arquivo": {
                        "type": "string",
                        "description": "Nome do arquivo a ser criado (sem a extensão .txt)"
                    },
                    "conteudo": {
                        "type": "string",
                        "description": "Conteúdo a ser escrito no arquivo"
                    }
                },
                "required": [
                    "nome_arquivo",
                    "conteudo"
                ]
            }
        }